## What is Regular Expression (Regex)?

Regular Expressions (**Regex**) are patterns used to **search, match, and manipulate text**.

They are extremely useful in **text preprocessing and Natural Language Processing (NLP)**, where we often need to detect patterns such as numbers, repeated characters, punctuation, or special symbols.

In Python, regex functionality is provided by the built‑in **`re` module**.

### Typical tasks with Regex include:

- Finding patterns in text  
- Replacing parts of text  
- Removing unwanted characters  
- Normalizing text before NLP processing


In [14]:
import re

# \d   → digit
# +    → one or more

text = "I bought 3 apples and 12 bananas"

numbers = re.findall(r"\d+", text)
print(numbers)



['3', '12']


In [15]:

# Hello world
# \s   → whitespace
# +    → one or more
text = "Hello     world"

clean = re.sub(r"\s+", " ", text)
print(clean)


Hello world


In [17]:

# (\S)     → capture any non-space character
# \1       → repeat the captured character
# {2,}     → at least 2 more times



text = "عاااااالی"

pattern = re.compile(r"(\S)\1{2,}")
print(pattern.findall(text))
print(re.sub(pattern, r"\1", text))
print(re.sub(pattern, r"\1\1", text))


['ا']
عالی
عاالی


In [4]:

# --- STEP 1: CHARACTER NORMALIZATION ---

# Create a translation table to convert Persian and Arabic digits into standard English digits.
# So both "۰۱۲۳۴۵۶۷۸۹" and "٠١٢٣٤٥٦٧٨٩" become "0123456789"
DIGIT_MAP = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")


## Invisible / Control Unicode Characters

The following Unicode characters are commonly removed during Persian/Arabic text normalization because they are invisible or affect text rendering and direction.

| Unicode | Name | Description |
|---|---|---|
| `\u200c` | Zero Width Non-Joiner (ZWNJ) | نیم‌فاصله در فارسی |
| `\u200d` | Zero Width Joiner (ZWJ) | اتصال اجباری کاراکترها |
| `\u200e` | Left-to-Right Mark (LRM) | علامت جهت متن چپ به راست |
| `\u200f` | Right-to-Left Mark (RLM) | علامت جهت متن راست به چپ |
| `\u202a` | Left-to-Right Embedding (LRE) | شروع متن چپ به راست |
| `\u202b` | Right-to-Left Embedding (RLE) | شروع متن راست به چپ |
| `\u202c` | Pop Directional Formatting (PDF) | پایان تنظیم جهت متن |
| `\u202d` | Left-to-Right Override (LRO) | اجبار جهت چپ به راست |
| `\u202e` | Right-to-Left Override (RLO) | اجبار جهت راست به چپ |
| `\ufeff` | Byte Order Mark (BOM) | نشانگر ترتیب بایت / کاراکتر مخفی ابتدای فایل |
| `\u00ad` | Soft Hyphen | خط تیره نرم (نامرئی) |
| `\u2060` | Word Joiner | جلوگیری از شکستن کلمات |
| `\u180e` | Mongolian Vowel Separator | جداکننده نامرئی قدیمی یونیکد |


In [18]:

# Compile a regex pattern to detect invisible/control Unicode characters:
# includes Zero-Width Non-Joiner (ZWNJ), direction marks, BOM, and soft hyphen.

# Compile regex for invisible/control characters
CONTROL_CHARS_RE = re.compile(r"[\u200c\u200d\u200e\u200f\u202a-\u202e\ufeff\u00ad\u2060\u180e]")

# Sample text containing invisible characters
text = "سلام\u200cدنیا\u200f امروز"

print("Original text:")
print(repr(text))

# Find invisible characters
found = CONTROL_CHARS_RE.findall(text)
print("\nDetected invisible characters:")
print(found)

# Remove them
clean_text = CONTROL_CHARS_RE.sub(" ", text)

print("\nCleaned text:")
print(repr(clean_text))


Original text:
'سلام\u200cدنیا\u200f امروز'

Detected invisible characters:
['\u200c', '\u200f']

Cleaned text:
'سلام دنیا  امروز'


- `[]` → character set  

- `^` → NOT (negation inside brackets)

- `0-9` → digits  

- `A-Z` → uppercase English letters  

- `a-z` → lowercase English letters  

- `\u0600-\u06FF` → Persian / Arabic Unicode range  

- `\s` → whitespace (space, tab, newline)


In [19]:
# Compile a regex pattern that keeps only:
# - English letters and digits (A–Z, a–z, 0–9)
# - Persian/Arabic characters (Unicode range \u0600–\u06FF)
# - Whitespace (\s)
# Everything else (emoji, Latin punctuation, symbols) can be removed or replaced.
KEEP_CHARS_RE = re.compile(r"[^0-9A-Za-z\u0600-\u06FF\s]")

In [30]:
import re
import pandas as pd
# from hazm import Normalizer, WordTokenizer, stopwords_list
from tqdm import tqdm

# Enable progress bars for Pandas
tqdm.pandas(desc="Processing")


# 1. Load the actual raw canonical dataset
df = pd.read_csv("comments_raw_canonical.csv")
df['text'] = df['text'].fillna("").astype(str)
df.head()

,doctor_id,source_file,comment_index_in_file,specialty,text,rate,label,date
0,100246,پوست_و_مو_doctor_337.json,0,متخصص پوست ، مو و زیبایی,بسیار حاذق,5.0,1,۱۴۰۳/۰۸/۰۴
1,100246,پوست_و_مو_doctor_337.json,1,متخصص پوست ، مو و زیبایی,زگیل زیر ناخن.تحت درمانم,5.0,1,۱۴۰۰/۰۲/۰۶
2,100246,پوست_و_مو_doctor_337.json,2,متخصص پوست ، مو و زیبایی,درود تشخیص درست خارش بدن ودرمان عالی,5.0,1,۱۴۰۴/۰۹/۲۸
3,100246,پوست_و_مو_doctor_337.json,3,متخصص پوست ، مو و زیبایی,بسیار با دقت و با حوصله,5.0,1,۱۴۰۴/۰۹/۲۷
4,100246,پوست_و_مو_doctor_337.json,4,متخصص پوست ، مو و زیبایی,تشخیص ایشان خیلی خوب بود ممنون,5.0,1,۱۴۰۴/۰۵/۲۷


In [31]:




# Compile a regex pattern to match one or more consecutive whitespace characters.
# Used to collapse multiple spaces into a single one during cleaning.
MULTI_SPACE = re.compile(r"\s+")

def step01_char_normalize(text: str) -> str:
    """Normalize Persian/Arabic text:
       - Remove control characters
       - Convert digits to English
       - Unify Arabic letters to Persian ones
       - Remove unwanted symbols
       - Normalize spaces
    """
    if not text:
        return ""

    # 1️⃣ Remove control/invisible Unicode characters
    t = CONTROL_CHARS_RE.sub(" ", text)

    # 2️⃣ Convert Persian and Arabic digits → English digits
    t = t.translate(DIGIT_MAP)

    # 3️⃣ Unify Arabic letters to Persian equivalents
    replacements = {
        "ي": "ی",
        "ك": "ک",
        "أ": "ا",
        "إ": "ا",
        "ؤ": "و",
        "ة": "ه",
        "ئ": "ی",
    }
    for old, new in replacements.items():
        t = t.replace(old, new)

    # 4️⃣ Remove any character not allowed (non-letter, non-digit, non-space)
    t = KEEP_CHARS_RE.sub(" ", t)

    # 5️⃣ Replace multiple spaces by a single space and trim whitespace
    t = MULTI_SPACE.sub(" ", t).strip()

    return t


# --- APPLY TO A DATAFRAME COLUMN ---
print("Running Step 1...")
df["step01_text"] = df["text"].progress_apply(step01_char_normalize)


# --- DEMONSTRATION SECTION ---
sample_1 = "پزشك بسیارمتبحرؤماهر است 🩺 شماره ٠١٢٣٤"
print("Example Before:", sample_1)
print("Example After :", step01_char_normalize(sample_1))

# You can test extra examples below 👇
more_samples = [
    "سال ۲۰۲۴ است",
    "كِتاب بسيار خوبيست",
    "مثال با نیم‌فاصله\u200Cها و اعداد ١٢٣٤٥٦",
    "سلام🌸 دنیا!!!"
]

for s in more_samples:
    print("\nBefore:", s)
    print("After :", step01_char_normalize(s))


Running Step 1...


Processing: 100%|██████████| 66465/66465 [00:00<00:00, 141178.91it/s]

Example Before: پزشك بسیارمتبحرؤماهر است 🩺 شماره ٠١٢٣٤
Example After : پزشک بسیارمتبحروماهر است شماره 01234

Before: سال ۲۰۲۴ است
After : سال 2024 است

Before: كِتاب بسيار خوبيست
After : کِتاب بسیار خوبیست

Before: مثال با نیم‌فاصله‌ها و اعداد ١٢٣٤٥٦
After : مثال با نیم فاصله ها و اعداد 123456

Before: سلام🌸 دنیا!!!
After : سلام دنیا


In [32]:
import re

# Regex pattern to detect elongated characters.
# (\S) captures any non-space character.
# \1{2,} means the same character repeats at least 2 more times (so total ≥3).
ELONG_RE = re.compile(r"(\S)\1{2,}")

text = "خیلی خییییییییییللللللللللییییییییی عاااااااااااااالللللللللیییی"

print("Original text:")
print(text)
print()

# ---- Case 1: Normalize to ONE character ----
# r"\1" means keep only the captured character once
one_repeat = ELONG_RE.sub(r"\1", text)

print("Normalize to 1 repetition:")
print(one_repeat)
print()

# ---- Case 2: Normalize to TWO characters ----
# r"\1\1" means repeat the captured character twice
two_repeat = ELONG_RE.sub(r"\1\1", text)

print("Normalize to 2 repetitions:")
print(two_repeat)
print()

# ---- Case 3: Normalize to FIVE characters ----
# r"\1\1\1\1\1" means repeat the captured character five times
five_repeat = ELONG_RE.sub(r"\1\1\1\1\1", text)

print("Normalize to 5 repetitions:")
print(five_repeat)


Original text:
خیلی خییییییییییللللللللللییییییییی عاااااااااااااالللللللللیییی

Normalize to 1 repetition:
خیلی خیلی عالی

Normalize to 2 repetitions:
خیلی خییللیی عااللیی

Normalize to 5 repetitions:
خیلی خیییییلللللییییی عااااالللللییییی


# The Step-by-Step Preprocessing Pipeline
In our EDA, we saw that our raw text contains mixed Persian/Arabic characters, English medical terms, emojis, and a lot of glued words. 

If we feed this directly into a Machine Learning model, the model will treat "درحال" and "در حال" as two completely different words. It will also focus on useless words like "عالی" instead of actual medical symptoms.

In this notebook, we will apply our **6-Step Pipeline** sequentially to our `66,465` real comments. Let's load the data!

In [34]:
import re
import pandas as pd
# from hazm import Normalizer, WordTokenizer, stopwords_list
from tqdm import tqdm

# Enable progress bars for Pandas
tqdm.pandas(desc="Processing")

# 1. Load the actual raw canonical dataset
df = pd.read_csv("comments_raw_canonical.csv")
df['text'] = df['text'].fillna("").astype(str)

print(f"Loaded {len(df):,} comments.")
# We will keep a small sample dataframe to print the before/after of each step clearly
sample_idx =[43491, 29934, 12182, 23583] # Specific interesting rows from the EDA reports

Loaded 66,465 comments.


### Step 1: Character Normalization
The very first step is to clean up the raw characters. Users type with different keyboards, which leads to different encodings for the same letters (e.g., Arabic `ي` vs Persian `ی`).

**What this step does:**
1. Removes invisible control characters (like Zero-Width Non-Joiners).
2. Converts all digits (Persian/Arabic) to standard English `0-9`.
3. Unifies Arabic character variants into standard Persian.
4. Removes completely non-standard characters (like Emojis).

*Enterprise Note: In our real data, this step modified over 10,000 rows (15.6%).*

In [36]:
# Create a translation table to convert Persian and Arabic digits into standard English digits.
DIGIT_MAP = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")

# Compile a regex pattern to detect invisible/control Unicode characters such as ZWNJ, direction marks, BOM, and soft hyphen.
CONTROL_CHARS_RE = re.compile(r"[\u200c\u200d\u200e\u200f\u202a-\u202e\ufeff\u00ad\u2060\u180e]")

# Compile a regex pattern to keep only English letters, digits, Arabic/Persian Unicode characters, and whitespace; everything else will be removable.
KEEP_CHARS_RE = re.compile(r"[^0-9A-Za-z\u0600-\u06FF\s]")

# Compile a regex pattern to match one or more consecutive whitespace characters for space normalization.
MULTI_SPACE = re.compile(r"\s+")

def step01_char_normalize(text: str) -> str:
    if not text: return ""
    t = CONTROL_CHARS_RE.sub(" ", text)
    t = t.translate(DIGIT_MAP)
    
    # Unify Arabic to Persian
    replacements = {"ي": "ی", "ك": "ک", "أ": "ا", "إ": "ا", "ؤ": "و", "ة": "ه", "ٱ": "ا", "ئ": "ی"}
    for old, new in replacements.items():
        t = t.replace(old, new)
        
    t = KEEP_CHARS_RE.sub(" ", t)
    t = MULTI_SPACE.sub(" ", t).strip()
    return t

# Apply to the entire dataset
print("Running Step 1...")
df['step01_text'] = df['text'].progress_apply(step01_char_normalize)


# --- DEMONSTRATION ---
# Notice the Arabic 'ؤ' and 'ك', the emoji, and the Arabic numbers
sample_1 = "پزشك بسیارمتبحرؤماهر است 🩺 شماره ٠١٢٣٤"
print("Example Before:", sample_1)
print("Example After :", step01_char_normalize(sample_1))

Running Step 1...


Processing: 100%|██████████| 66465/66465 [00:00<00:00, 159070.06it/s]

Example Before: پزشك بسیارمتبحرؤماهر است 🩺 شماره ٠١٢٣٤
Example After : پزشک بسیارمتبحروماهر است شماره 01234


### Step 2: Pre-Hazm Clean (Punctuation & Elongation)
Before we pass text to an advanced NLP library, we need to strip away excessive punctuation and "elongations."
Users often stretch words to show emotion (e.g., `عاااااااااااااالی`). We use Regular Expressions to reduce any character repeated 3 or more times down to just 2 times.

*Enterprise Note: This step affected nearly 4,000 rows, safely removing Persian punctuation (`،؛؟!`) while preserving decimal dots in numbers.*

In [37]:
# Compile a regex pattern to detect elongated characters where the same non-space character repeats 3 or more times (used to normalize emphasis like "عاااالی").
ELONG_RE = re.compile(r"(\S)\1{2,}")

# Compile a regex pattern to match common Persian punctuation marks so they can be removed or normalized.
PERS_PUNCT_RE = re.compile(r"[،؛؟!٪«»""]+")

# Compile a regex pattern to match dots that are NOT between digits (keeps decimal numbers like 3.14 but removes other periods).
DOT_SAFE_RE = re.compile(r"(?<!\d)\.(?!\d)")


def step02_pre_hazm_clean(text: str) -> str:
    if not text: return ""
    t = ELONG_RE.sub(r"\1\1", text) # Reduces 3+ repetitions to 2
    t = PERS_PUNCT_RE.sub(" ", t)
    t = DOT_SAFE_RE.sub(" ", t)
    t = MULTI_SPACE.sub(" ", t).strip()
    return t

print("Running Step 2...")
df['step02_text'] = df['step01_text'].progress_apply(step02_pre_hazm_clean)

# --- DEMONSTRATION ---
# Taken directly from the extreme EDA cases!
sample_2 = "خیلی خیییییییییللللللللللییییییییی عاااااااااااااالللللللللیییی ،،،،،"
print("Example Before:", sample_2)
print("Example After :", step02_pre_hazm_clean(sample_2))

Running Step 2...


Processing: 100%|██████████| 66465/66465 [00:00<00:00, 164375.97it/s]

Example Before: خیلی خیییییییییللللللللللییییییییی عاااااااااااااالللللللللیییی ،،،،،
Example After : خیلی خییللیی عااللیی


### Step 3: Formal Normalization using `Hazm`
Now the text is clean enough for `Hazm`, the standard library for Persian NLP.
Hazm enforces proper spacing rules. For example, it attaches prefixes and suffixes correctly (e.g., fixing `می روم` to `می‌روم`).

*Enterprise Note: In the actual dataset, Hazm successfully fixed spacing in **12,536 rows (18.9%)**, massively improving our word boundaries.*

In [38]:
from hazm import Normalizer, WordTokenizer, stopwords_list

hazm_normalizer = Normalizer(
    correct_spacing=True,
    remove_diacritics=True,
    remove_specials_chars=False,
    decrease_repeated_chars=False,
    persian_style=True,
    persian_numbers=False
)

def step03_hazm_normalize(text: str) -> str:
    if not text: return ""
    return hazm_normalizer.normalize(text)

print("Running Step 3 (This might take a minute)...")
df['step03_text'] = df['step02_text'].progress_apply(step03_hazm_normalize)

df.loc[sample_idx,['step02_text', 'step03_text']].tail(2)

# --- DEMONSTRATION ---
# Notice how Hazm separates "3سال" and elegantly fixes the verb prefix in "میکردم"
sample_3 = "من 3سال پیش دیسک گردن داشتم که عمل میکردم"
print("Example Before:", sample_3)
print("Example After :", step03_hazm_normalize(sample_3))

ModuleNotFoundError: No module named 'numpy.char'

### Step 4: Domain-Specific "De-gluing"
General NLP libraries don't know the quirks of specific platforms. In our medical dataset, patients frequently glued phrases together, which Hazm missed. 
For instance, `درحال` (instead of `در حال`) and `خداروشکر` appeared hundreds of times! We use hard-coded Regex rules to split these specific domain typos.


In [28]:
DEGLUE_RULES =[
    (re.compile(r"(?<!\S)درحال(?!\S)"), "در حال"),
    (re.compile(r"(?<!\S)باسلام(?!\S)"), "با سلام"),
    (re.compile(r"(?<!\S)خداروشکر(?!\S)"), "خدا رو شکر"),
    (re.compile(r"(?<!\S)فوقالعاده(?!\S)"), "فوق العاده"),
    (re.compile(r"(?<!\S)اززحماتشون(?!\S)"), "از زحماتشون"),
    (re.compile(r"(?<!\S)اماایشان(?!\S)"), "اما ایشان"),
    (re.compile(r"(?<!\S)ازبچگی(?!\S)"), "از بچگی"),
    (re.compile(r"(?<!\S)هردکتری(?!\S)"), "هر دکتری"),
    (re.compile(r"(?<!\S)وسپس(?!\S)"), "و سپس"),
    (re.compile(r"(?<!\S)برطرفشد(?!\S)"), "برطرف شد"),
]

def step04_domain_deglue(text: str) -> str:
    t = text
    for pat, repl in DEGLUE_RULES:
        t = pat.sub(repl, t)
    return MULTI_SPACE.sub(" ", t).strip()

print("Running Step 4...")
df['step04_text'] = df['step03_text'].progress_apply(step04_domain_deglue)


# --- DEMONSTRATION ---
sample_4 = "باسلام، من درحال درمانم و خداروشکر بهترم. پزشک فوقالعاده ای است."
# Note: we pass it through step 2 to remove the comma first to simulate the pipeline state
sample_4_state = step02_pre_hazm_clean(sample_4) 

print("Example Before:", sample_4_state)
print("Example After :", step04_domain_deglue(sample_4_state))

Running Step 4...


Processing: 100%|██████████| 66465/66465 [00:00<00:00, 118990.51it/s]

Example Before: باسلام من درحال درمانم و خداروشکر بهترم پزشک فوقالعاده ای است
Example After : با سلام من در حال درمانم و خدا رو شکر بهترم پزشک فوق العاده ای است


### Step 5: Tokenization & Base Stopwords (Protecting Negations)
It's time to break the text into individual words, called **Tokens**. 
We also remove "Stopwords" (common words like `از`, `به`, `که`). 

**Crucial NLP Insight:** Standard stopword lists often contain negation words (`نه`, `نیست`). If we delete these, "I am not satisfied" becomes "I am satisfied." We must explicitly protect negations!

In [29]:
tokenizer = WordTokenizer(separate_emoji=False, replace_links=False, replace_ids=False)

# Build standard stopwords but protect negations
standard_stopwords = set(stopwords_list())
negation_keep = {"نه", "نخیر", "نیست", "نمی", "نمي", "نبود", "نشد", "بدون"}
standard_stopwords = standard_stopwords - negation_keep

def step05_tokenize(text: str) -> list:
    if not text: return[]
    tokens = tokenizer.tokenize(text)
    # Keep tokens >= 2 chars, and exclude standard stopwords
    return[t for t in tokens if len(t) >= 2 and t not in standard_stopwords]

print("Running Step 5 (Tokenization)...")
df['step05_tokens'] = df['step04_text'].progress_apply(step05_tokenize)

# --- DEMONSTRATION ---
sample_5 = "من از مطب دکتر اصلا راضی نیستم و مراجعه نمی کنم"
print("Example Before:", sample_5)
print("Example After :", step05_tokenize(sample_5))
print("\nNotice how 'از' and 'و' were removed, but 'نیستم' and 'نمی' were kept!")

Running Step 5 (Tokenization)...


Processing: 100%|██████████| 66465/66465 [00:00<00:00, 145474.90it/s]

Example Before: من از مطب دکتر اصلا راضی نیستم و مراجعه نمی کنم
Example After : ['مطب', 'دکتر', 'اصلا', 'راضی', 'نیستم', 'مراجعه', 'نمی']

Notice how 'از' and 'و' were removed, but 'نیستم' and 'نمی' were kept!


### Step 6: Medical Domain Stopwords & Safety Filters
We are building a search engine to retrieve doctors based on medical queries. Our EDA showed that words like `عالی` (Excellent) and `دکتر` (Doctor) appear over 15,000 times! They provide zero medical value for search.

However, we must **never** delete high-risk medical words (`درد`, `جراحی`) or English acronyms (`mri`, `ivf`), even if they are short. We also use Bigrams to remove multi-word praises like `فوق العاده`.

In [30]:
MEDICAL_STOPWORDS = {
    "عالی", "خوب", "بهترین", "محشر", "راضی", "رضایت", "ناراضی",
    "مرسی", "ممنون", "تشکر", "سپاس", "قدردانی", "دکتر", "پزشک", "خانم", "آقا", "آقای", 
    "مطب", "منشی", "پرسنل", "پذیرش", "نوبت", "نوبتدهی", "وقت", "معطلی",
    "شلوغ", "هزینه", "ویزیت", "حوصله", "اخلاق", "حاذق", "سلام", "واقعا", 
    "مراجعه", "نتیجه", "هستم", "هستن", "هستند", "داشتم", "بودم", "بودن", "شد", "شدم", "کردم"
}
# Safety filters
MEDICAL_ALLOWLIST_LATIN = {"acl", "mcl", "pcl", "mri", "ct", "prp", "hpv", "pcnl", "ivf", "iui"}
HIGH_RISK_KEEP = {"درد", "دارو", "درمان", "تشخیص", "عمل", "جراحی", "بی", "فوق", "کار"}
MEDICAL_BIGRAM_STOPWORDS = {("فوق", "العاده"), ("بی", "نظیر")}

effective_medical_stopwords = MEDICAL_STOPWORDS - HIGH_RISK_KEEP

def step06_medical_stopwords(tokens: list) -> str:
    kept_tokens =[]
    i = 0
    while i < len(tokens):
        t1 = tokens[i].lower()
        
        if t1 in MEDICAL_ALLOWLIST_LATIN:
            kept_tokens.append(t1)
            i += 1
            continue
            
        if i + 1 < len(tokens):
            t2 = tokens[i + 1].lower()
            if (t1, t2) in MEDICAL_BIGRAM_STOPWORDS:
                i += 2
                continue
                
        if t1 in effective_medical_stopwords:
            i += 1
            continue
            
        kept_tokens.append(tokens[i])
        i += 1
        
    return " ".join(kept_tokens) # Return as a clean string

print("Running Step 6...")
df['final_preprocessed_text'] = df['step05_tokens'].progress_apply(step06_medical_stopwords)

# --- DEMONSTRATION ---
sample_6_tokens =['دکتر', 'بسیار', 'عالی', 'بود', 'برای', 'درد', 'کمر', 'و', 'mri', 'رفتم', 'فوق', 'العاده', 'بود']
print("Example Before:", sample_6_tokens)
print("Example After :", step06_medical_stopwords(sample_6_tokens))
print("\nNotice how 'دکتر', 'عالی' and 'فوق العاده' are gone, but 'درد', 'کمر' and 'mri' survived!")

Running Step 6...


Processing: 100%|██████████| 66465/66465 [00:00<00:00, 484158.19it/s]

Example Before: ['دکتر', 'بسیار', 'عالی', 'بود', 'برای', 'درد', 'کمر', 'و', 'mri', 'رفتم', 'فوق', 'العاده', 'بود']
Example After : بسیار بود برای درد کمر و mri رفتم بود

Notice how 'دکتر', 'عالی' and 'فوق العاده' are gone, but 'درد', 'کمر' and 'mri' survived!


### Finalization: Filtering Noise & Saving the Data
We've successfully transformed our messy data into clean, highly relevant medical tokens!
Notice how generic praise has disappeared, but words like `دیسک` (disc), `شانه` (shoulder), and `عمل` (surgery) remain.

To prepare for our TF-IDF Search Engine, we will drop any row that became completely empty (because it contained only noise/stopwords), as well as the fake `عدم رضایت` placeholders.


In [31]:
# 1. Identify the system placeholders from our EDA
df["is_placeholder"] = (df["label"] == -1) & (df["text"].str.strip() == "عدم رضایت")

# 2. Identify rows that became completely empty after all 6 steps
df['is_empty'] = df['final_preprocessed_text'].str.strip() == ""

# 3. Filter the dataframe
retriever_df = df[(~df['is_empty']) & (~df['is_placeholder'])].copy()

print(f"Original Comments: {len(df):,}")
print(f"Comments remaining for Retriever: {len(retriever_df):,}")
print(f"Noise comments dropped: {len(df) - len(retriever_df):,}")

# Keep only the columns needed for Notebook 3
retriever_df = retriever_df[['doctor_id', 'final_preprocessed_text', 'rate', 'label', 'date']]

# Save to CSV
out_file = "comments_for_tfidf_retriever.csv"
retriever_df.to_csv(out_file, index=False, encoding="utf-8-sig")

print(f"\nClean dataset saved to '{out_file}'!")

Original Comments: 66,465
Comments remaining for Retriever: 52,800
Noise comments dropped: 13,665

Clean dataset saved to 'comments_for_tfidf_retriever.csv'!
